# CS345 - E-Commerce Session Purchase Classification Using Ensemble Machine Learning 

## **คนที่ 1**

### นำไฟล์ Dataset เข้า Databricks

In [0]:
%pip install --upgrade kaggle

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import subprocess

# ดึง Kaggle API Token จาก Databricks Secrets
os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get(
    scope="kaggle",
    key="api_token"
)

print("Kaggle token loaded successfully!")

In [0]:
dataset_id = "mkechinov/ecommerce-behavior-data-from-multi-category-store"

volume_path = "/Volumes/workspace/default/ecommerce_data"

# ดาวน์โหลดเฉพาะไฟล์เดือนตุลาคม
command = [
    "kaggle",
    "datasets", "download",
    "-d", dataset_id,
    "-f", "2019-Oct.csv",
    "-p", volume_path,
    "--unzip"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
else:
    print("Download completed!")

In [0]:
# ดาวน์โหลดไฟล์เดือนพฤศจิกายน
command = [
    "kaggle",
    "datasets", "download",
    "-d", dataset_id,
    "-f", "2019-Nov.csv",
    "-p", volume_path,
    "--unzip"
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
else:
    print("Download completed!")

In [0]:
display(
    dbutils.fs.ls(volume_path)
)

### อ่าน CSV ด้วย PySpark

In [0]:
oct_path = "/Volumes/workspace/default/ecommerce_data/2019-Oct.csv"

df_oct = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(oct_path)
)

print("October Schema:")
df_oct.printSchema()

In [0]:
nov_path = "/Volumes/workspace/default/ecommerce_data/2019-Nov.csv"

df_nov = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(nov_path)
)

print("November Schema:")
df_nov.printSchema()

In [0]:
df_all = df_oct.unionByName(df_nov)

print("Combined Schema:")
df_all.printSchema()

In [0]:
display(df_all.limit(10))

### เตรียมข้อมูลก่อนสร้าง Session

In [0]:
from pyspark.sql import functions as F

# กำหนด Timezone เป็น UTC ตามข้อมูลต้นฉบับ
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_prepared = (
    df_all
    # แปลง event_time เป็น timestamp
    .withColumn(
        "event_ts",
        F.to_timestamp(F.col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'")
    )
    # สร้าง Session Key
    .withColumn(
        "session_key",
        F.concat(F.col("user_id"), F.lit("||"), F.col("user_session"))
    )
    # แปลง price เป็นตัวเลข 
    .withColumn("price", F.col("price").cast("double"))
    # ลบแถวที่ไม่มีเวลา, user_id หรือ user_session
    .filter(
        F.col("event_ts").isNotNull() &
        F.col("user_id").isNotNull() &
        F.col("user_session").isNotNull() &
        (F.trim(F.col("user_session")) != "")
    )
    # เก็บเฉพาะ ต.ค. - พ.ย. 2019
    .filter((F.col("event_ts") >= "2019-10-01") & (F.col("event_ts") < "2019-12-01"))
    # ลบ event_time เดิม และลบแถวซ้ำ
    .drop("event_time")
    .dropDuplicates()
)

In [0]:
df_prepared.select(
    "event_ts",
    "user_id",
    "user_session",
    "session_key",
    "event_type",
    "price"
).show(10, truncate=False)

In [0]:
df_prepared.printSchema()

### บันทึกเป็น Delta Table
รันครั้งเดียวพอ ต่อไปอ่านจาก table แทน CSV จะเร็วกว่ามาก

In [0]:
df_prepared.write.mode("overwrite").saveAsTable("workspace.default.ecommerce_events_clean")

print("Saved ecommerce_events_clean!")

In [0]:
# อ่านกลับมาจาก table
events = spark.table("workspace.default.ecommerce_events_clean")

print("จำนวน events:", events.count())

In [0]:
# ดูว่ามี event_type อะไรบ้าง (ถ้าไม่มี remove_from_cart ให้แจ้งคน 2)
events.groupBy("event_type").count().show()

### สร้าง Session + กำหนด Prediction Point (Session Start + 5 นาที)

In [0]:
sessions = (
    events
    .groupBy("session_key")
    .agg(
        F.min("user_id").alias("user_id"),
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end"),
        F.count("*").alias("n_events_total")
    )
    # Prediction Point = Session Start + 5 นาที
    .withColumn("prediction_point", F.col("session_start") + F.expr("INTERVAL 5 MINUTES"))
)

sessions.show(5, truncate=False)

In [0]:
# บันทึก sessions ไว้ก่อน จะได้ไม่ต้องคำนวณใหม่
sessions.write.mode("overwrite").saveAsTable("workspace.default.sessions_base")

sessions = spark.table("workspace.default.sessions_base")

print("จำนวน sessions:", sessions.count())

### แปะ Prediction Point ให้แต่ละ event
แล้วดูว่า event นั้นเกิด **ก่อน** หรือ **หลัง** Prediction Point

In [0]:
events_pp = events.join(
    sessions.select("session_key", "prediction_point"),
    on="session_key",
    how="inner"
)

# before_pp = 1 ถ้า event เกิดภายใน 5 นาทีแรก
events_pp = events_pp.withColumn(
    "before_pp",
    F.when(F.col("event_ts") <= F.col("prediction_point"), 1).otherwise(0)
)

# is_buy = 1 ถ้าเป็น event purchase
events_pp = events_pp.withColumn(
    "is_buy",
    F.when(F.col("event_type") == "purchase", 1).otherwise(0)
)

events_pp.select("session_key", "event_ts", "prediction_point", "event_type", "before_pp", "is_buy").show(10, truncate=False)

### สร้าง Label (Target)
- `buy_before_pp` = ซื้อภายใน 5 นาทีแรกหรือไม่ (จะตัดทิ้ง)
- `is_purchase` = ซื้อหลัง 5 นาทีแรกหรือไม่ (**นี่คือ Target**)
- `n_events_window` = จำนวน event (ที่ไม่ใช่ purchase) ใน 5 นาทีแรก

In [0]:
labels = (
    events_pp
    .groupBy("session_key")
    .agg(
        F.max(F.when((F.col("before_pp") == 1) & (F.col("is_buy") == 1), 1).otherwise(0)).alias("buy_before_pp"),
        F.max(F.when((F.col("before_pp") == 0) & (F.col("is_buy") == 1), 1).otherwise(0)).alias("is_purchase"),
        F.sum(F.when((F.col("before_pp") == 1) & (F.col("is_buy") == 0), 1).otherwise(0)).alias("n_events_window")
    )
)

sessions_labeled = sessions.join(labels, on="session_key", how="inner")

sessions_labeled.show(5, truncate=False)

In [0]:
# ดูว่าต้องตัด session ทิ้งกี่อัน (เอาไปเขียนใน Report)
print("Sessions ทั้งหมด:", sessions_labeled.count())
print("ซื้อภายใน 5 นาทีแรก (ตัดทิ้ง):", sessions_labeled.filter(F.col("buy_before_pp") == 1).count())

In [0]:
# ตัด session ที่ซื้อไปแล้วภายใน 5 นาทีแรก
sessions_labeled = sessions_labeled.filter(F.col("buy_before_pp") == 0)

# ตัด session ที่ไม่มี event ให้สร้าง feature
sessions_labeled = sessions_labeled.filter(F.col("n_events_window") > 0)

print("Sessions ที่เหลือ:", sessions_labeled.count())

### Time-based Split (ใช้ session_start) แล้วบันทึก
- Train: 1–21 Oct
- Validation: 22–31 Oct
- Test: 1–30 Nov

In [0]:
sessions_labeled = sessions_labeled.withColumn(
    "split",
    F.when(F.col("session_start") < "2019-10-22", "train")
     .when(F.col("session_start") < "2019-11-01", "validation")
     .otherwise("test")
)

# บันทึก sessions_labeled (ทุกคนในทีมใช้ table นี้)
sessions_labeled.write.mode("overwrite").saveAsTable("workspace.default.sessions_labeled")

sessions_labeled = spark.table("workspace.default.sessions_labeled")

sessions_labeled.groupBy("split").count().show()

### เตรียม events ใน 5 นาทีแรก ส่งให้คน 2
เอาเฉพาะ event ก่อน Prediction Point และไม่เอา purchase

In [0]:
events_window = events_pp.filter(
    (F.col("before_pp") == 1) &
    (F.col("is_buy") == 0)
)

# เอาเฉพาะ session ที่ผ่านการกรองแล้ว และติด split มาด้วย
events_window = events_window.join(
    sessions_labeled.select("session_key", "session_start", "split"),
    on="session_key",
    how="inner"
)

events_window = events_window.select(
    "session_key", "split", "session_start", "prediction_point", "event_ts",
    "event_type", "product_id", "category_id", "category_code", "brand", "price"
)

events_window.write.mode("overwrite").saveAsTable("workspace.default.events_window")

events_window = spark.table("workspace.default.events_window")

print("Saved events_window!")

### ตรวจ Data Leakage
ข้อ 1–3 ต้องได้ **0** และข้อ 4 ช่วงเวลาของแต่ละ split ต้องไม่ทับกัน

In [0]:
# 1) มี purchase หลุดเข้ามาใน events_window ไหม
print("purchase ใน events_window:", events_window.filter(F.col("event_type") == "purchase").count())

# 2) มี event หลัง prediction point หลุดเข้ามาไหม
print("event หลัง prediction point:", events_window.filter(F.col("event_ts") > F.col("prediction_point")).count())

# 3) มี session ซ้ำไหม
print("session ซ้ำ:", sessions_labeled.groupBy("session_key").count().filter(F.col("count") > 1).count())

In [0]:
# 4) ช่วงเวลาของแต่ละ split ต้องไม่ทับกัน
sessions_labeled.groupBy("split").agg(
    F.min("session_start").alias("first_session"),
    F.max("session_start").alias("last_session")
).show(truncate=False)

### สรุปผล (ใช้ใน Report)

In [0]:
# จำนวน session และสัดส่วนที่ซื้อ ในแต่ละ split
sessions_labeled.groupBy("split").agg(
    F.count("*").alias("sessions"),
    F.sum("is_purchase").alias("purchase_sessions"),
    F.round(F.avg("is_purchase") * 100, 2).alias("purchase_rate_pct")
).show()

In [0]:
# จำนวน events ต่อ session ใน 5 นาทีแรก
sessions_labeled.groupBy("n_events_window").count().orderBy("n_events_window").show(20)

## คนที่2